In [ ]:
# =========================================================
# CELL 1: TRAIN DIRECT CUT + SHIP MODELS
# EQUAL-WEIGHT MODEL SELECTION
# SAVE MODELS + WRITE STREAMLIT APP
# =========================================================

# 1. Import required libraries for file handling, model training, evaluation, and saving
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.linear_model import LinearRegression

# -------------------------
# CONFIG
# -------------------------

# 2. Define file paths and sheet names
EXCEL_PATH = r"D:\Savidhu_OneDrive\OneDrive - Hirdaramani Group\Projects\Cut to Ship Prediction Model\Final Report\Cut to Ship Modified Cleaned.xlsx"
TRAIN_SHEET = "Train"
TEST_SHEET = "Test"
ALL_SHEET = "All"

BASE_DIR = os.path.dirname(EXCEL_PATH)

CUT_MODEL_PATH = os.path.join(BASE_DIR, "direct_cut_qty_model.pkl")
SHIP_MODEL_PATH = os.path.join(BASE_DIR, "direct_ship_qty_model.pkl")
META_PATH = os.path.join(BASE_DIR, "direct_deploy_meta.pkl")
PREDICTIONS_PATH = os.path.join(BASE_DIR, "direct_deploy_test_predictions.xlsx")
METRICS_PATH = os.path.join(BASE_DIR, "direct_model_comparison.xlsx")
APP_PATH = os.path.join(BASE_DIR, "streamlit_app_direct.py")

# 3. Define target columns
TARGET_CUT = "Cut Qty"
TARGET_SHIP = "Ship Qty"

# 4. Define base input columns used in the model and UI
BASE_INPUT_COLS = [
    "Year", "Calling Name", "Div", "Season", "Garment item type", "Unit",
    "Operation", "Month", "Week", "Type", "Operation 2", "Set garment",
    "Pcs", "Order Qty"
]

# 5. Define reason columns
REASON_COLS = [f"Metric {chr(i)}" for i in range(ord("A"), ord("S") + 1)]

# 6. Define UI columns
UI_COLS = BASE_INPUT_COLS.copy()

# -------------------------
# HELPERS
# -------------------------

# 7. Convert values to numeric safely
def to_num(series):
    s = (
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.strip()
    )
    s = s.replace({"": np.nan, "nan": np.nan, "None": np.nan, "N/A": np.nan})
    return pd.to_numeric(s, errors="coerce")

# 8. Clean text values and fill missing categories
def clean_text(series):
    s = series.fillna("Unknown").astype(str).str.strip()
    s = s.replace({"": "Unknown", "nan": "Unknown", "None": "Unknown", "N/A": "Unknown"})
    return s

# 9. Safely sum only columns that exist
def safe_sum(df, cols):
    cols = [c for c in cols if c in df.columns]
    if not cols:
        return pd.Series(0.0, index=df.index)
    return df[cols].fillna(0).sum(axis=1)

# 10. Identify non-zero target values for percentage-based metrics
def get_non_zero_mask(y_true):
    y_true = np.asarray(y_true, dtype=float)
    return np.abs(y_true) > 1e-9

# 11. Calculate MAPE percentage
def mape_pct(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = get_non_zero_mask(y_true)
    if mask.sum() == 0:
        return np.nan
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / np.abs(y_true[mask]))) * 100)

# 12. Convert MAPE into accuracy percentage
def accuracy_from_mape(y_true, y_pred):
    mape = mape_pct(y_true, y_pred)
    if np.isnan(mape):
        return np.nan
    return float(np.clip(100.0 - mape, 0.0, 100.0))

# 13. Calculate percentage of predictions within a tolerance band
def within_pct(y_true, y_pred, pct):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = get_non_zero_mask(y_true)
    if mask.sum() == 0:
        return np.nan
    return float(np.mean(np.abs(y_true[mask] - y_pred[mask]) / np.abs(y_true[mask]) <= pct / 100.0) * 100)

# 14. Create one evaluation row for a model
def build_metric_row(y_true, y_pred, model_name, target_name):
    return {
        "Target": target_name,
        "Model": model_name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "MAPE %": mape_pct(y_true, y_pred),
        "Accuracy %": accuracy_from_mape(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
        "Within ±5%": within_pct(y_true, y_pred, 5),
        "Within ±10%": within_pct(y_true, y_pred, 10),
        "Rows": len(y_true),
    }

# 15. Add equal-weight combined score across all evaluation metrics
def add_equal_weight_score(results_df):
    df_score = results_df.copy()

    for col in ["MAE", "MAPE %"]:
        col_min = df_score[col].min()
        col_max = df_score[col].max()
        if pd.isna(col_min) or pd.isna(col_max) or col_max == col_min:
            df_score[f"{col}_Score"] = 1.0
        else:
            df_score[f"{col}_Score"] = (col_max - df_score[col]) / (col_max - col_min)

    for col in ["Accuracy %", "R2", "Within ±5%", "Within ±10%"]:
        col_min = df_score[col].min()
        col_max = df_score[col].max()
        if pd.isna(col_min) or pd.isna(col_max) or col_max == col_min:
            df_score[f"{col}_Score"] = 1.0
        else:
            df_score[f"{col}_Score"] = (df_score[col] - col_min) / (col_max - col_min)

    df_score["Combined Score"] = (
        df_score["MAE_Score"]
        + df_score["MAPE %_Score"]
        + df_score["Accuracy %_Score"]
        + df_score["R2_Score"]
        + df_score["Within ±5%_Score"]
        + df_score["Within ±10%_Score"]
    ) / 6.0

    df_score = df_score.sort_values(
        ["Combined Score", "MAPE %", "Within ±10%", "Within ±5%", "R2", "MAE"],
        ascending=[False, True, False, False, False, True]
    ).reset_index(drop=True)

    return df_score

# 16. Load and clean one sheet for modelling
def prepare_df(sheet_name):
    df = pd.read_excel(EXCEL_PATH, sheet_name=sheet_name)
    df.columns = [str(c).strip() for c in df.columns]

    needed = BASE_INPUT_COLS + [TARGET_CUT, TARGET_SHIP]
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in {sheet_name}: {missing}")

    keep_cols = list(dict.fromkeys(BASE_INPUT_COLS + [TARGET_CUT, TARGET_SHIP] + [c for c in REASON_COLS if c in df.columns]))
    df = df[keep_cols].copy()

    for col in ["Pcs", "Order Qty", TARGET_CUT, TARGET_SHIP]:
        df[col] = to_num(df[col])

    for col in [c for c in REASON_COLS if c in df.columns]:
        df[col] = to_num(df[col]).fillna(0)

    for col in BASE_INPUT_COLS:
        if col not in ["Pcs", "Order Qty"]:
            df[col] = clean_text(df[col])

    df["Pcs"] = df["Pcs"].fillna(0)

    df = df.dropna(subset=["Order Qty", TARGET_CUT, TARGET_SHIP]).copy()
    df = df[(df["Order Qty"] > 0) & (df[TARGET_CUT] >= 0) & (df[TARGET_SHIP] >= 0)].copy()

    return df

# 17. Create grouped reason-based features
def add_reason_features(df):
    df = df.copy()

    damage_cols = ["Metric A", "Metric B", "Metric C", "Metric J"]
    transfer_cols = ["Metric R", "Metric S"]
    sample_cols = ["Metric K", "Metric F", "Metric E"]
    quality_cols = ["Metric G", "Metric H", "Metric O", "Metric P", "Metric Q"]
    reconciliation_cols = ["Metric M", "Metric N"]

    present_reason_cols = [c for c in REASON_COLS if c in df.columns]

    df["Damage_Total"] = safe_sum(df, damage_cols)
    df["Transfer_Total"] = safe_sum(df, transfer_cols)
    df["Sample_Total"] = safe_sum(df, sample_cols)
    df["Quality_Total"] = safe_sum(df, quality_cols)
    df["Reconciliation_Total"] = safe_sum(df, reconciliation_cols)
    df["Total_Reason_Qty"] = safe_sum(df, present_reason_cols)

    if present_reason_cols:
        df["Reason_Count_NonZero"] = (df[present_reason_cols].fillna(0) > 0).sum(axis=1)
    else:
        df["Reason_Count_NonZero"] = 0

    df["Has_Any_Reason"] = (df["Total_Reason_Qty"] > 0).astype(int)
    df["Has_Transfer"] = (df["Transfer_Total"] > 0).astype(int)
    df["Has_Damage"] = (df["Damage_Total"] > 0).astype(int)
    df["Has_Reconciliation_Issue"] = (df["Reconciliation_Total"] > 0).astype(int)

    return df

# 18. Create interaction features between important categorical columns
def add_interaction_features(df):
    df = df.copy()

    for col in ["Year", "Calling Name", "Div", "Season", "Garment item type", "Unit", "Operation", "Month", "Type", "Operation 2"]:
        if col in df.columns:
            df[col] = clean_text(df[col])

    df["Year_Month"] = df["Year"] + "_" + df["Month"]
    df["Div_Unit"] = df["Div"] + "_" + df["Unit"]
    df["Season_Garment"] = df["Season"] + "_" + df["Garment item type"]
    df["CallingName_Garment"] = df["Calling Name"] + "_" + df["Garment item type"]
    df["Operation_Type"] = df["Operation"] + "_" + df["Type"]
    df["Operation_Operation2"] = df["Operation"] + "_" + df["Operation 2"]

    return df

# 19. Create simple ratio-based feature
def add_ratio_features(df):
    df = df.copy()
    eps = 1e-9
    df["Pcs_per_OrderQty"] = df["Pcs"] / (df["Order Qty"] + eps)
    return df

# 20. Create frequency encoding features using train data only
def add_frequency_features(train_df, test_df):
    train_df = train_df.copy()
    test_df = test_df.copy()

    freq_maps = {}
    mapping = {
        "Year": "Year_Freq",
        "Calling Name": "Calling_Name_Freq",
        "Garment item type": "Garment_Type_Freq",
        "Div": "Div_Freq",
        "Unit": "Unit_Freq",
        "Operation": "Operation_Freq",
    }

    for src_col, feat_col in mapping.items():
        freq_map = train_df[src_col].value_counts().to_dict()
        train_df[feat_col] = train_df[src_col].map(freq_map).astype(float)
        test_df[feat_col] = test_df[src_col].map(freq_map).fillna(0).astype(float)
        freq_maps[feat_col] = freq_map

    return train_df, test_df, freq_maps

# 21. Build historical lookup tables with backoff levels
def build_lookup_tables(df, key_cols, feat_cols):
    full = df.groupby(key_cols, dropna=False)[feat_cols].mean().reset_index()
    backoff1_keys = [c for c in key_cols if c != "Calling Name"]
    backoff1 = df.groupby(backoff1_keys, dropna=False)[feat_cols].mean().reset_index()
    backoff2_keys = [c for c in backoff1_keys if c != "Season"]
    backoff2 = df.groupby(backoff2_keys, dropna=False)[feat_cols].mean().reset_index()

    return {
        "key_cols": key_cols,
        "backoff1_keys": backoff1_keys,
        "backoff2_keys": backoff2_keys,
        "full": full,
        "backoff1": backoff1,
        "backoff2": backoff2,
        "global_mean": df[feat_cols].mean().to_dict(),
        "feat_cols": feat_cols
    }

# 22. Retrieve historical lookup values with fallback logic
def lookup_behavior(row_dict, lookups):
    feat_cols = lookups["feat_cols"]

    for table_name, key_name in [("full", "key_cols"), ("backoff1", "backoff1_keys"), ("backoff2", "backoff2_keys")]:
        table = lookups[table_name]
        keys = lookups[key_name]

        mask = np.ones(len(table), dtype=bool)
        for k in keys:
            mask &= (table[k].astype(str).values == str(row_dict[k]))

        match = table.loc[mask]
        if len(match) > 0:
            return {f: float(match.iloc[0][f]) for f in feat_cols}

    return {f: float(lookups["global_mean"].get(f, 0.0)) for f in feat_cols}

# 23. Add historical lookup-based features back into dataframe
def add_lookup_features(df, lookups):
    rows = []
    for _, row in df.iterrows():
        row_dict = {k: row[k] for k in lookups["key_cols"]}
        rows.append(lookup_behavior(row_dict, lookups))
    out = pd.DataFrame(rows, index=df.index).add_prefix("Hist_")
    return pd.concat([df, out], axis=1)

# 24. Define candidate models to compare
def make_model_candidates():
    return {
        "LinearRegression": LinearRegression(),
        "RandomForest": RandomForestRegressor(
            n_estimators=300, max_depth=18, random_state=42, n_jobs=-1
        ),
        "GradientBoosting": GradientBoostingRegressor(
            n_estimators=250, learning_rate=0.05, max_depth=3, random_state=42
        ),
        "ExtraTrees": ExtraTreesRegressor(
            n_estimators=350, max_depth=18, random_state=42, n_jobs=-1
        ),
    }

# 25. Create OneHotEncoder with version-safe settings
def make_onehot():
    from sklearn.preprocessing import OneHotEncoder
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

# 26. Build preprocessing and model pipeline
def build_pipeline(cat_cols, num_cols, model):
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="constant", fill_value=0))
            ]), num_cols),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
                ("onehot", make_onehot())
            ]), cat_cols)
        ],
        remainder="drop"
    )

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    return pipe

# 27. Train and evaluate all candidate models for one target
def evaluate_target_models(X_train, y_train, X_test, y_test, cat_cols, num_cols, target_name):
    candidates = make_model_candidates()
    results = []
    fitted = {}
    predictions = {}

    for model_name, model in candidates.items():
        pipe = build_pipeline(cat_cols, num_cols, model)
        pipe.fit(X_train, y_train)
        pred = pipe.predict(X_test)
        pred = np.clip(pred, 0, None)

        results.append(build_metric_row(y_test, pred, model_name, target_name))
        fitted[model_name] = pipe
        predictions[model_name] = pred

    results_df = pd.DataFrame(results)
    results_df = add_equal_weight_score(results_df)

    best_model_name = results_df.loc[0, "Model"]
    best_pipe = fitted[best_model_name]
    best_pred = predictions[best_model_name]

    return results_df, best_model_name, best_pipe, best_pred

# -------------------------
# LOAD DATA
# -------------------------

# 28. Load cleaned train, test, and full datasets
train_df = prepare_df(TRAIN_SHEET)
test_df = prepare_df(TEST_SHEET)
all_df = prepare_df(ALL_SHEET)

# 29. Add reason-based engineered features
train_df = add_reason_features(train_df)
test_df = add_reason_features(test_df)
all_df = add_reason_features(all_df)

# 30. Add interaction features
train_df = add_interaction_features(train_df)
test_df = add_interaction_features(test_df)
all_df = add_interaction_features(all_df)

# 31. Add ratio features
train_df = add_ratio_features(train_df)
test_df = add_ratio_features(test_df)
all_df = add_ratio_features(all_df)

# 32. Add frequency encoding features
train_df, test_df, freq_maps = add_frequency_features(train_df, test_df)

# 33. Define lookup key columns
KEY_COLS = [
    "Year", "Calling Name", "Div", "Season", "Garment item type",
    "Unit", "Operation", "Month", "Type", "Operation 2"
]

# 34. Define columns used to build historical lookup features
LOOKUP_FEATURES = [
    TARGET_CUT, TARGET_SHIP,
    "Order Qty", "Pcs_per_OrderQty",
    "Total_Reason_Qty", "Damage_Total", "Transfer_Total", "Quality_Total"
]

# 35. Build and apply historical lookup features
lookups = build_lookup_tables(train_df, KEY_COLS, LOOKUP_FEATURES)
train_df = add_lookup_features(train_df, lookups)
test_df = add_lookup_features(test_df, lookups)
all_df = add_lookup_features(all_df, lookups)

# 36. Define final feature list used for modelling
FEATURE_COLS = [
    "Year", "Calling Name", "Div", "Season", "Garment item type", "Unit",
    "Operation", "Month", "Week", "Type", "Operation 2", "Set garment",
    "Pcs", "Order Qty",

    "Year_Month", "Div_Unit", "Season_Garment", "CallingName_Garment",
    "Operation_Type", "Operation_Operation2",

    "Pcs_per_OrderQty",

    "Year_Freq", "Calling_Name_Freq", "Garment_Type_Freq",
    "Div_Freq", "Unit_Freq", "Operation_Freq",

    "Reason_Count_NonZero", "Total_Reason_Qty", "Damage_Total", "Transfer_Total",
    "Sample_Total", "Quality_Total", "Reconciliation_Total",
    "Has_Any_Reason", "Has_Transfer", "Has_Damage", "Has_Reconciliation_Issue",

    "Hist_Cut Qty", "Hist_Ship Qty", "Hist_Order Qty", "Hist_Pcs_per_OrderQty",
    "Hist_Total_Reason_Qty", "Hist_Damage_Total", "Hist_Transfer_Total", "Hist_Quality_Total"
]

# 37. Keep only features that exist in the training dataset
FEATURE_COLS = [c for c in FEATURE_COLS if c in train_df.columns]

# 38. Split features into train and test inputs
X_train = train_df[FEATURE_COLS].copy()
X_test = test_df[FEATURE_COLS].copy()

# 39. Identify categorical and numeric feature columns
cat_cols = [c for c in FEATURE_COLS if train_df[c].dtype == "object"]
num_cols = [c for c in FEATURE_COLS if c not in cat_cols]

# 40. Prepare target arrays for Cut Qty and Ship Qty
y_train_cut = train_df[TARGET_CUT].astype(float).values
y_test_cut = test_df[TARGET_CUT].astype(float).values

y_train_ship = train_df[TARGET_SHIP].astype(float).values
y_test_ship = test_df[TARGET_SHIP].astype(float).values

# -------------------------
# CUT QTY
# -------------------------

# 41. Train and evaluate all models for Cut Qty
cut_results, best_cut_model_name, best_cut_pipe, best_cut_pred = evaluate_target_models(
    X_train, y_train_cut, X_test, y_test_cut, cat_cols, num_cols, "Cut Qty"
)

# 42. Print Cut Qty model comparison and selected model
print("===== CUT QTY MODEL PERFORMANCE =====")
print(cut_results[[
    "Target", "Model", "MAE", "MAPE %", "Accuracy %", "R2", "Within ±5%", "Within ±10%", "Combined Score", "Rows"
]].to_string(index=False))
print("\nChosen model for Cut Qty:", best_cut_model_name)

# -------------------------
# SHIP QTY
# -------------------------

# 43. Train and evaluate all models for Ship Qty
ship_results, best_ship_model_name, best_ship_pipe, best_ship_pred = evaluate_target_models(
    X_train, y_train_ship, X_test, y_test_ship, cat_cols, num_cols, "Ship Qty"
)

# 44. Print Ship Qty model comparison and selected model
print("\n===== SHIP QTY MODEL PERFORMANCE =====")
print(ship_results[[
    "Target", "Model", "MAE", "MAPE %", "Accuracy %", "R2", "Within ±5%", "Within ±10%", "Combined Score", "Rows"
]].to_string(index=False))
print("\nChosen model for Ship Qty:", best_ship_model_name)

# -------------------------
# OVERALL
# -------------------------

# 45. Combine Cut Qty and Ship Qty comparison results
overall_metrics = pd.concat([cut_results, ship_results], ignore_index=True)

# 46. Print overall model performance table
print("\n===== OVERALL MODEL PERFORMANCE =====")
print(overall_metrics[[
    "Target", "Model", "MAE", "MAPE %", "Accuracy %", "R2", "Within ±5%", "Within ±10%", "Combined Score", "Rows"
]].to_string(index=False))

# -------------------------
# SAVE PREDICTIONS
# -------------------------

# 47. Save best model predictions on test data
pred_output = test_df.copy()
pred_output["Pred_CutQty"] = np.round(best_cut_pred, 0)
pred_output["Pred_ShipQty"] = np.round(best_ship_pred, 0)
pred_output.to_excel(PREDICTIONS_PATH, index=False)

# -------------------------
# SAVE METRICS
# -------------------------

# 48. Save model comparison tables to Excel
with pd.ExcelWriter(METRICS_PATH, engine="openpyxl") as writer:
    cut_results.to_excel(writer, sheet_name="Cut_Model_Comparison", index=False)
    ship_results.to_excel(writer, sheet_name="Ship_Model_Comparison", index=False)
    overall_metrics.to_excel(writer, sheet_name="Overall_Model_Comparison", index=False)

# -------------------------
# SAVE MODELS
# -------------------------

# 49. Save best trained pipelines
joblib.dump(best_cut_pipe, CUT_MODEL_PATH)
joblib.dump(best_ship_pipe, SHIP_MODEL_PATH)

# 50. Store allowed dropdown values for Streamlit UI
allowed_values = {}
for col in UI_COLS:
    if col not in ["Order Qty", "Pcs"]:
        vals = sorted(all_df[col].dropna().astype(str).unique().tolist())
        allowed_values[col] = vals if vals else ["Unknown"]

# 51. Build metadata dictionary for deployment
meta = {
    "base_input_cols": BASE_INPUT_COLS,
    "ui_cols": UI_COLS,
    "feature_cols": FEATURE_COLS,
    "allowed_values": allowed_values,
    "freq_maps": freq_maps,
    "lookups": lookups,
    "cut_mape_pct": float(cut_results.loc[cut_results["Model"] == best_cut_model_name, "MAPE %"].iloc[0]),
    "ship_mape_pct": float(ship_results.loc[ship_results["Model"] == best_ship_model_name, "MAPE %"].iloc[0]),
    "best_cut_model_name": best_cut_model_name,
    "best_ship_model_name": best_ship_model_name,
    "target_cut": TARGET_CUT,
    "target_ship": TARGET_SHIP,
}

# 52. Save metadata file
joblib.dump(meta, META_PATH)

# 53. Print saved output file locations
print("\nSaved files:")
print(CUT_MODEL_PATH)
print(SHIP_MODEL_PATH)
print(META_PATH)
print(PREDICTIONS_PATH)
print(METRICS_PATH)

# -------------------------
# WRITE STREAMLIT APP
# -------------------------

# 54. Write full Streamlit prediction app code as a Python string
app_code = r'''
import os
import joblib
import numpy as np
import pandas as pd
import streamlit as st

BASE_DIR = r"D:\Savidhu_OneDrive\OneDrive - Hirdaramani Group\Projects\Cut to Ship Prediction Model\Final Report"
CUT_MODEL_PATH = os.path.join(BASE_DIR, "direct_cut_qty_model.pkl")
SHIP_MODEL_PATH = os.path.join(BASE_DIR, "direct_ship_qty_model.pkl")
META_PATH = os.path.join(BASE_DIR, "direct_deploy_meta.pkl")

cut_pipe = joblib.load(CUT_MODEL_PATH)
ship_pipe = joblib.load(SHIP_MODEL_PATH)
meta = joblib.load(META_PATH)

BASE_INPUT_COLS = meta["base_input_cols"]
UI_COLS = meta["ui_cols"]
FEATURE_COLS = meta["feature_cols"]
ALLOWED_VALUES = meta["allowed_values"]
FREQ_MAPS = meta["freq_maps"]
LOOKUPS = meta["lookups"]

CUT_MAPE_PCT = float(meta.get("cut_mape_pct", 5.0))
SHIP_MAPE_PCT = float(meta.get("ship_mape_pct", 5.0))

def clean_text_value(x):
    if x is None:
        return "Unknown"
    x = str(x).strip()
    if x == "" or x.lower() in ["nan", "none", "n/a"]:
        return "Unknown"
    return x

def safe_float_value(x, default=0.0):
    if x is None:
        return default
    s = str(x).strip()
    if s == "":
        return default
    try:
        return float(s.replace(",", ""))
    except Exception:
        return default

def safe_lookup_freq(freq_map, key):
    return float(freq_map.get(key, 0))

def lookup_behavior(row_dict, lookups):
    feat_cols = lookups["feat_cols"]

    for table_name, key_name in [("full", "key_cols"), ("backoff1", "backoff1_keys"), ("backoff2", "backoff2_keys")]:
        table = lookups[table_name]
        keys = lookups[key_name]

        mask = np.ones(len(table), dtype=bool)
        for k in keys:
            mask &= (table[k].astype(str).values == str(row_dict[k]))

        match = table.loc[mask]
        if len(match) > 0:
            return {f: float(match.iloc[0][f]) for f in feat_cols}

    return {f: float(lookups["global_mean"].get(f, 0.0)) for f in feat_cols}

def pct_text(x):
    return f"{x * 100:.1f}%"

def metric_card(label, value, accent_class="blue"):
    return f"""
    <div class="metric-card {accent_class}">
        <div class="metric-label">{label}</div>
        <div class="metric-value">{value}</div>
    </div>
    """

st.set_page_config(page_title="Cut & Ship Prediction Model", page_icon="📊", layout="wide")

st.markdown("""
<style>
    .stApp { background: linear-gradient(180deg, #f4f7fb 0%, #eef3f8 100%); }
    .block-container { padding-top: 2rem; padding-bottom: 2rem; max-width: 1250px; }
    .hero-box { background: linear-gradient(135deg, #0f2747 0%, #163a63 60%, #1d4c7f 100%); padding: 28px 32px; border-radius: 20px; color: white; margin-bottom: 1.2rem; }
    .hero-title { font-size: 2.1rem; font-weight: 700; margin-bottom: 0.2rem; }
    .section-box { background: #ffffff; border-radius: 18px; padding: 20px 22px 16px 22px; box-shadow: 0 10px 24px rgba(31, 45, 61, 0.08); border: 1px solid #dbe5f0; margin-bottom: 1.2rem; }
    .section-title { font-size: 1.2rem; font-weight: 700; color: #102a43; margin-bottom: 0.8rem; }
    .stButton > button { background: linear-gradient(135deg, #0f4c81 0%, #166088 100%); color: white; border: none; border-radius: 12px; padding: 0.7rem 1.6rem; font-weight: 700; }
    .metric-card { background: #ffffff; border-radius: 18px; padding: 18px 20px; border: 1px solid #dbe5f0; min-height: 118px; margin-bottom: 14px; position: relative; overflow: hidden; }
    .metric-card::before { content: ""; position: absolute; top: 0; left: 0; width: 100%; height: 5px; }
    .metric-card.blue::before { background: linear-gradient(90deg, #0f4c81, #1b6ca8); }
    .metric-card.teal::before { background: linear-gradient(90deg, #0d6e6e, #1b9aaa); }
    .metric-card.gold::before { background: linear-gradient(90deg, #b8860b, #d4a017); }
    .metric-label { font-size: 0.95rem; font-weight: 600; color: #486581; margin-bottom: 0.55rem; }
    .metric-value { font-size: 2rem; font-weight: 800; color: #102a43; line-height: 1.1; }
    .results-title { font-size: 1.35rem; font-weight: 700; color: #102a43; margin-bottom: 0.85rem; }
    .range-box { background: #f8fbff; border: 1px solid #dbe5f0; border-radius: 14px; padding: 14px 16px; margin-top: 8px; }
</style>
""", unsafe_allow_html=True)

st.markdown("""
<div class="hero-box">
    <div class="hero-title">Cut &amp; Ship Prediction Model</div>
</div>
""", unsafe_allow_html=True)

st.markdown('<div class="section-box">', unsafe_allow_html=True)
st.markdown('<div class="section-title">Enter Order Details</div>', unsafe_allow_html=True)

cols = st.columns(3)
inputs = {}

for i, col in enumerate(UI_COLS):
    with cols[i % 3]:
        if col == "Order Qty":
            inputs[col] = st.number_input(col, min_value=1.0, step=1.0, value=1000.0)
        elif col == "Pcs":
            raw_val = st.text_input(col, value="", placeholder="")
            inputs[col] = safe_float_value(raw_val, default=0.0)
        else:
            options = [""] + ALLOWED_VALUES.get(col, ["Unknown"])
            selected = st.selectbox(col, options=options)
            inputs[col] = selected if str(selected).strip() != "" else "Unknown"

predict_clicked = st.button("Predict")
st.markdown('</div>', unsafe_allow_html=True)

if predict_clicked:
    try:
        eps = 1e-9
        order_qty = float(inputs["Order Qty"])

        row = {}
        for col in BASE_INPUT_COLS:
            if col == "Order Qty":
                row[col] = float(inputs[col])
            elif col == "Pcs":
                row[col] = safe_float_value(inputs[col], default=0.0)
            else:
                row[col] = clean_text_value(inputs[col])

        row["Year_Month"] = f"{row['Year']}_{row['Month']}"
        row["Div_Unit"] = f"{row['Div']}_{row['Unit']}"
        row["Season_Garment"] = f"{row['Season']}_{row['Garment item type']}"
        row["CallingName_Garment"] = f"{row['Calling Name']}_{row['Garment item type']}"
        row["Operation_Type"] = f"{row['Operation']}_{row['Type']}"
        row["Operation_Operation2"] = f"{row['Operation']}_{row['Operation 2']}"
        row["Pcs_per_OrderQty"] = row["Pcs"] / (row["Order Qty"] + eps)

        row["Reason_Count_NonZero"] = 0.0
        row["Total_Reason_Qty"] = 0.0
        row["Damage_Total"] = 0.0
        row["Transfer_Total"] = 0.0
        row["Sample_Total"] = 0.0
        row["Quality_Total"] = 0.0
        row["Reconciliation_Total"] = 0.0
        row["Has_Any_Reason"] = 0
        row["Has_Transfer"] = 0
        row["Has_Damage"] = 0
        row["Has_Reconciliation_Issue"] = 0

        row["Year_Freq"] = safe_lookup_freq(FREQ_MAPS.get("Year_Freq", {}), row["Year"])
        row["Calling_Name_Freq"] = safe_lookup_freq(FREQ_MAPS.get("Calling_Name_Freq", {}), row["Calling Name"])
        row["Garment_Type_Freq"] = safe_lookup_freq(FREQ_MAPS.get("Garment_Type_Freq", {}), row["Garment item type"])
        row["Div_Freq"] = safe_lookup_freq(FREQ_MAPS.get("Div_Freq", {}), row["Div"])
        row["Unit_Freq"] = safe_lookup_freq(FREQ_MAPS.get("Unit_Freq", {}), row["Unit"])
        row["Operation_Freq"] = safe_lookup_freq(FREQ_MAPS.get("Operation_Freq", {}), row["Operation"])

        lookup_row = {
            "Year": row["Year"],
            "Calling Name": row["Calling Name"],
            "Div": row["Div"],
            "Season": row["Season"],
            "Garment item type": row["Garment item type"],
            "Unit": row["Unit"],
            "Operation": row["Operation"],
            "Month": row["Month"],
            "Type": row["Type"],
            "Operation 2": row["Operation 2"],
        }

        hist = lookup_behavior(lookup_row, LOOKUPS)
        row["Hist_Cut Qty"] = hist.get("Cut Qty", 0.0)
        row["Hist_Ship Qty"] = hist.get("Ship Qty", 0.0)
        row["Hist_Order Qty"] = hist.get("Order Qty", 0.0)
        row["Hist_Pcs_per_OrderQty"] = hist.get("Pcs_per_OrderQty", 0.0)
        row["Hist_Total_Reason_Qty"] = hist.get("Total_Reason_Qty", 0.0)
        row["Hist_Damage_Total"] = hist.get("Damage_Total", 0.0)
        row["Hist_Transfer_Total"] = hist.get("Transfer_Total", 0.0)
        row["Hist_Quality_Total"] = hist.get("Quality_Total", 0.0)

        numeric_defaults = {
            "Pcs", "Order Qty", "Pcs_per_OrderQty",
            "Year_Freq", "Calling_Name_Freq", "Garment_Type_Freq", "Div_Freq", "Unit_Freq", "Operation_Freq",
            "Reason_Count_NonZero", "Total_Reason_Qty", "Damage_Total", "Transfer_Total", "Sample_Total",
            "Quality_Total", "Reconciliation_Total", "Has_Any_Reason", "Has_Transfer", "Has_Damage",
            "Has_Reconciliation_Issue", "Hist_Cut Qty", "Hist_Ship Qty", "Hist_Order Qty", "Hist_Pcs_per_OrderQty",
            "Hist_Total_Reason_Qty", "Hist_Damage_Total", "Hist_Transfer_Total", "Hist_Quality_Total"
        }

        for col in FEATURE_COLS:
            if col not in row:
                row[col] = 0.0 if col in numeric_defaults else "Unknown"

        X = pd.DataFrame([row])[FEATURE_COLS].copy()

        cut_qty_pred = float(cut_pipe.predict(X)[0])
        ship_qty_pred = float(ship_pipe.predict(X)[0])

        cut_qty_pred = max(0.0, cut_qty_pred)
        ship_qty_pred = max(0.0, ship_qty_pred)

        cut_low = cut_qty_pred * (1 - CUT_MAPE_PCT / 100.0)
        cut_high = cut_qty_pred * (1 + CUT_MAPE_PCT / 100.0)
        ship_low = ship_qty_pred * (1 - SHIP_MAPE_PCT / 100.0)
        ship_high = ship_qty_pred * (1 + SHIP_MAPE_PCT / 100.0)

        cut_ship = ship_qty_pred / (cut_qty_pred + eps)
        order_ship = ship_qty_pred / (order_qty + eps)
        order_cut = cut_qty_pred / (order_qty + eps)

        st.markdown('<div class="section-box">', unsafe_allow_html=True)
        st.markdown('<div class="results-title">Prediction Results</div>', unsafe_allow_html=True)

        r1, r2, r3 = st.columns(3)
        with r1:
            st.markdown(metric_card("Predicted Cut Qty", f"{int(round(cut_qty_pred)):,}", "blue"), unsafe_allow_html=True)
        with r2:
            st.markdown(metric_card("Predicted Ship Qty", f"{int(round(ship_qty_pred)):,}", "teal"), unsafe_allow_html=True)
        with r3:
            st.markdown(metric_card("Order Qty", f"{int(round(order_qty)):,}", "gold"), unsafe_allow_html=True)

        r4, r5, r6 = st.columns(3)
        with r4:
            st.markdown(metric_card("Cut / Ship", pct_text(cut_ship), "blue"), unsafe_allow_html=True)
        with r5:
            st.markdown(metric_card("Order / Ship", pct_text(order_ship), "teal"), unsafe_allow_html=True)
        with r6:
            st.markdown(metric_card("Order / Cut", pct_text(order_cut), "gold"), unsafe_allow_html=True)

        st.markdown('<div class="range-box">', unsafe_allow_html=True)
        st.markdown("**Estimated Prediction Range Based on Historical Model Error**")
        st.markdown(
            f"- **Cut Qty Range:** {int(round(cut_low)):,} to {int(round(cut_high)):,} (±{CUT_MAPE_PCT:.1f}%)\n"
            f"- **Ship Qty Range:** {int(round(ship_low)):,} to {int(round(ship_high)):,} (±{SHIP_MAPE_PCT:.1f}%)"
        )
        st.markdown('</div>', unsafe_allow_html=True)

        st.markdown('</div>', unsafe_allow_html=True)

    except Exception as e:
        st.error(f"Prediction failed: {e}")
'''

# 55. Save Streamlit app code into a Python file
with open(APP_PATH, "w", encoding="utf-8") as f:
    f.write(app_code)

# 56. Print Streamlit app file location
print("\nApp written to:")
print(APP_PATH)

In [ ]:
# =========================================================
# CELL 2: RUN STREAMLIT APP
# =========================================================

# 1. Import libraries needed to run the app and open it in the browser
import os
import sys
import time
import subprocess
import webbrowser

# 2. Define app folder path, app file path, port number, and local URL
BASE_DIR = r"D:\Savidhu_OneDrive\OneDrive - Hirdaramani Group\Projects\Cut to Ship Prediction Model\Final Report"
APP_PATH = os.path.join(BASE_DIR, "streamlit_app_direct.py")
PORT = 8515
URL = f"http://localhost:{PORT}"

# 3. Start the Streamlit app as a new process
print("Launching Streamlit app...")
proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", APP_PATH, "--server.port", str(PORT)],
    cwd=BASE_DIR
)

# 4. Wait a few seconds for the app to start
time.sleep(4)

# 5. Open the Streamlit app automatically in the default web browser
webbrowser.open(URL)

# 6. Print the app URL and process ID
print("Use this URL:")
print(URL)
print("PID:", proc.pid)

In [ ]:
# =========================================================
# CELL 3: MODEL VS COMPANY METHOD
# =========================================================

# 1. Import required libraries for calculations, file loading, and evaluation
import numpy as np
import pandas as pd
import os
import joblib
from sklearn.metrics import mean_absolute_error

# -----------------------------
# CONFIG
# -----------------------------

# 2. Define base folder path, input files, company rule, and target column
BASE_DIR = r"D:\Savidhu_OneDrive\OneDrive - Hirdaramani Group\Projects\Cut to Ship Prediction Model\Final Report"

TEST_FILE = os.path.join(BASE_DIR, "direct_deploy_test_predictions.xlsx")
META_PATH = os.path.join(BASE_DIR, "direct_deploy_meta.pkl")

COMPANY_FACTOR = 1.03
TARGET_CUT = "Cut Qty"

# -----------------------------
# LOAD DATA
# -----------------------------

# 3. Load prediction file and deployment metadata
df = pd.read_excel(TEST_FILE)
meta = joblib.load(META_PATH)

# 4. Read saved best model name from metadata
best_cut_model_name = meta.get("best_cut_model_name", "Model")

# -----------------------------
# Actuals and predictions
# -----------------------------

# 5. Extract actual Cut Qty values, model predictions, and company method predictions
y_true = df[TARGET_CUT].astype(float).values
model_pred = df["Pred_CutQty"].astype(float).values
company_pred = df["Order Qty"].astype(float).values * COMPANY_FACTOR

# -----------------------------
# Order-level comparison
# -----------------------------

# 6. Calculate absolute error for model and company method on each order
model_error = np.abs(y_true - model_pred)
company_error = np.abs(y_true - company_pred)

# 7. Count how many orders each method performs better on
model_better = int(np.sum(model_error < company_error))
company_better = int(np.sum(model_error > company_error))
ties = int(np.sum(model_error == company_error))
total_orders = len(y_true)

# 8. Convert order-level counts into percentages
model_better_pct = (model_better / total_orders) * 100 if total_orders > 0 else np.nan
company_better_pct = (company_better / total_orders) * 100 if total_orders > 0 else np.nan
ties_pct = (ties / total_orders) * 100 if total_orders > 0 else np.nan

# -----------------------------
# SAFE METRICS
# -----------------------------

# 9. Calculate MAPE safely by ignoring zero actual values
def mape_pct(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.abs(y_true) > 1e-9
    if mask.sum() == 0:
        return np.nan
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / np.abs(y_true[mask]))) * 100)

# 10. Convert MAPE into accuracy percentage
def accuracy_from_mape(y_true, y_pred):
    mape = mape_pct(y_true, y_pred)
    if np.isnan(mape):
        return np.nan
    return float(np.clip(100.0 - mape, 0.0, 100.0))

# -----------------------------
# Metrics
# -----------------------------

# 11. Create list to store comparison results
comparison_rows = []

# 12. Add model performance metrics
comparison_rows.append({
    "Method": f"Best Model ({best_cut_model_name})",
    "MAE": mean_absolute_error(y_true, model_pred),
    "MAPE %": mape_pct(y_true, model_pred),
    "Accuracy %": accuracy_from_mape(y_true, model_pred)
})

# 13. Add company method performance metrics
comparison_rows.append({
    "Method": "Company Method (103% of Order Qty)",
    "MAE": mean_absolute_error(y_true, company_pred),
    "MAPE %": mape_pct(y_true, company_pred),
    "Accuracy %": accuracy_from_mape(y_true, company_pred)
})

# 14. Convert comparison results into dataframe
comparison_df = pd.DataFrame(comparison_rows)

# -----------------------------
# Equal-weight scoring
# -----------------------------

# 15. Create copy of results for scoring
score_df = comparison_df.copy()

# 16. Convert lower-is-better metrics into normalized scores
for col in ["MAE", "MAPE %"]:
    col_min = score_df[col].min()
    col_max = score_df[col].max()
    if col_max == col_min:
        score_df[f"{col}_Score"] = 1.0
    else:
        score_df[f"{col}_Score"] = (col_max - score_df[col]) / (col_max - col_min)

# 17. Convert higher-is-better metric into normalized score
col = "Accuracy %"
col_min = score_df[col].min()
col_max = score_df[col].max()
if col_max == col_min:
    score_df[f"{col}_Score"] = 1.0
else:
    score_df[f"{col}_Score"] = (score_df[col] - col_min) / (col_max - col_min)

# 18. Calculate final equal-weight score across MAE, MAPE, and Accuracy
score_df["Final Score"] = (
    score_df["MAE_Score"] +
    score_df["MAPE %_Score"] +
    score_df["Accuracy %_Score"]
) / 3.0

# 19. Sort methods from best to worst
score_df = score_df.sort_values(
    ["Final Score", "MAPE %", "MAE", "Accuracy %"],
    ascending=[False, True, True, False]
).reset_index(drop=True)

# -----------------------------
# PRINT RESULTS
# -----------------------------

# 20. Print overall metric comparison
print("===== MODEL VS COMPANY METHOD =====")
print(score_df[[
    "Method", "MAE", "MAPE %", "Accuracy %", "Final Score"
]].to_string(index=False))

# 21. Print order-level comparison summary
print("\n===== ORDER-LEVEL PERFORMANCE =====")
print(f"Total Orders: {total_orders}")
print(f"Model better on: {model_better} orders ({model_better_pct:.2f}%)")
print(f"Company better on: {company_better} orders ({company_better_pct:.2f}%)")
print(f"Ties: {ties} orders ({ties_pct:.2f}%)")

# 22. Print final best method
print("\n===== FINAL RESULT =====")
print(f"Better overall method: {score_df.iloc[0]['Method']}")